## RAG Day 3 — IT Help Desk Assistant

### Connecting the Day 2 vector store to an LLM for a real chat loop

Day 2 gave us a persisted Chroma store full of embedded chunks from the
SUNY New Paltz TeamDynamix knowledge base. Today we connect that store to
an LLM and get an actual question-answering pipeline working end to end.

The shape of everything we do today is:

**retrieve -> format into prompt -> generate**

No query rewriting, no reranking, no evaluation metrics yet — that's Day 5
and Day 4 respectively. This is the minimal viable RAG pipeline, using
LangChain's abstractions end-to-end (`retriever`, `llm`, `invoke`).

**Stack:** `gemini-embedding-001` (must match Day 2 ingestion) +
`gemini-2.5-flash-lite` for chat.

In [1]:
from pathlib import Path
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

load_dotenv(override=True)

MODEL = "gemini-2.5-flash-lite"
DB_NAME = "vector_db"

d:\mrsha\Projects\HawkEye\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Reconnect to the persisted store.
# IMPORTANT: must use the SAME embedding model used to build it in Day 2
# (gemini-embedding-001) or retrieval quality silently breaks.

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

print(f"Vectorstore loaded with {vectorstore._collection.count():,} vectors")

Vectorstore loaded with 7,676 vectors


### Set up the two key LangChain objects: retriever and llm

- `retriever` wraps the vectorstore's similarity search behind a standard
  `.invoke(question)` interface.
- `temperature=0` means "always pick the most likely token" — appropriate
  for a help desk assistant that should be factual and consistent, not
  creative.

In [3]:
retriever = vectorstore.as_retriever()
llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0)

### Sanity check both objects independently before wiring them together

In [4]:
# Swap this for a question you know is answerable from your knowledge base
retriever.invoke("How do I reset my password?")

[Document(id='4c8f1e9a-9834-40d6-8297-feffc1c2735f', metadata={'source': 'knowledge-base/Accounts-Access-Security/Forgot-Password--Password-Reset-24520.md', 'url': 'https://newpaltz.teamdynamix.com/TDClient/1905/Portal/KB/Article/24520/Forgot-Password-Password-Reset', 'title': 'Forgot Password / Password Reset', 'doc_type': 'Accounts-Access-Security'}, page_content='If you have forgotten your New Paltz username or password you can use our self-service password reset tool by going to the\xa0[Self-service\xa0Reset Link.](https://webapps.newpaltz.edu/npcuid/recover.php)\n\nThe self-service tool will only work if:\n\n* We have a valid, external email address (i.e. a non-“newpaltz.edu” email address) on file for you.\n* You have already set up a “security question” so your identity can be verified.\n\nIf either\xa0of the above criteria are not met and you are unable to use the self-service tool, you can still reset your password by one of the following options:'),
 Document(id='f3d42417-0ce

In [5]:
llm.invoke("How do I reset my password?")

AIMessage(content='I can help you with that! To reset your password, I need a little more information. **What website or service are you trying to reset your password for?**\n\nOnce you tell me the specific service, I can provide you with the exact steps.\n\nIn general, the process usually involves these common steps:\n\n1.  **Go to the login page:** Navigate to the website or app where you want to log in.\n2.  **Look for a "Forgot Password" or "Reset Password" link:** This is usually located near the login fields (username/email and password).\n3.  **Enter your account identifier:** You\'ll likely be asked to enter the email address or username associated with your account.\n4.  **Check your email:** The service will send you an email with instructions or a link to reset your password.\n5.  **Follow the instructions:** Click on the link in the email and follow the prompts to create a new password.\n\n**Please tell me which service you need help with, and I\'ll give you more specific i

## Time to put this together!

Write the system prompt template with a `{context}` placeholder, then
the `answer_question` function that does retrieve -> format -> generate.

In [10]:
SYSTEM_PROMPT_TEMPLATE = """
You are an internal knowledge assistant for IT Help Desk technicians at SUNY New Paltz.
You do not talk to students, faculty, or staff directly — you are only ever talking to
the technician, who is the middleman handling the actual customer interaction.

The technician will describe what the customer is asking or the issue they're facing
(e.g. "customer is asking how to reset their password"). Give the technician a direct,
straightforward answer: the relevant steps, procedure, or policy, in plain language —
like a coworker quickly telling another coworker what to do. Don't script out exact
phrases to relay to the customer unless the technician specifically asks for wording.
Keep it tight — no unnecessary preamble, headers, or restating the question.

If you don't know the answer or it isn't in the context, say so rather than guessing —
don't leave the technician to relay a wrong answer.

Context:
{context}
"""

In [11]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [12]:
answer_question("Customer is asking how to reset their password", [])

'1. Go to the Self-service Reset Link.\n2. Enter the user\'s NPCUID and click "Continue."\n3. If a recovery email is on file, click "Send Reset E-mail."\n4. If no recovery email is on file, ask the customer if they have access to a Gmail, Yahoo, or Hotmail account associated with their account. If they do, follow the instructions on "Recovery Email - Add / Replace."\n5. If the self-service tool doesn\'t work (e.g., no external email on file or security question not set up), direct them to contact the IT Service Desk via email at servicedesk@newpaltz.edu, by phone at (845) 257-4357, or in person at Humanities 103.'

## Wire it up to Gradio

Gotcha carried over from Day 1: **do not** pass `type="messages"` to
`gr.ChatInterface(...)` — Gradio 6.20.0 (your installed version) throws
`TypeError: unexpected keyword argument 'type'`. Just call it plain.

In [13]:
gr.ChatInterface(answer_question).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## What we accomplished today

- Reconnected to the Day 2 Chroma store using the matching embedding model.
- Built the two core LangChain objects: `retriever` and `llm`.
- Wrote the minimal RAG loop: retrieve top-k chunks -> stuff into a system
  prompt -> generate an answer.
- Got it running in a real Gradio chat window.

**What this doesn't do yet (by design):** no query rewriting for multi-turn
context, no reranking, no dual retrieval, no quantitative evaluation. Those
are Day 4 (measure it) and Day 5 (improve it). Per the fast-track plan, we're
not pausing here to manually probe its limitations — Day 4's actual metrics
will tell us where it's weak.